# < EOS > Punctuation Classification Pipeline

This notebook implements a comprehensive machine learning pipeline to distinguish between punctuation symbols whether they are ending a sentence or not. The approach combines advanced feature engineering, Bayesian hyperparameter optimization, and ensemble learning techniques.

### Problem Overview
- **Task**: Binary Classification
- **Challenge**: ...
- **Solution**: Feature engineering + Logistic Regression + LinearSVC + Stacking with calibration

In [ ]:
from pathlib import Path
import random

file_path = Path("data/raw/UD_English-EWT/en_ewt-ud-train.sent_split")

text = file_path.read_text(encoding="utf-8")

print("Number of characters:", len(text))
print("Number of <EOS> tags:", text.count("<EOS>"))
print("\nFirst 2000 chars:\n")
print(text[:2000])

In [ ]:
# ==============================================================================
# IMPORTS AND DEPENDENCIES
# ==============================================================================
# Core data manipulation and numerical computing libraries
import numpy as np
import pandas as pd

# Scikit-learn pipeline and preprocessing tools
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

# Ensemble methods and model calibration
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import StackingClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from lightgbm import LGBMClassifier

# Visualization libraries
import seaborn as sns
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

import pandas as pd

pd.set_option("display.max_columns", None)   
pd.set_option("display.width", None)        
pd.set_option("display.max_colwidth", None)  

In [40]:
import importlib
import utils.preprocessing as prep_mod

importlib.reload(prep_mod)
print(dir(prep_mod))

['Path', 'SentenceSplitPreprocessor', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'pd', 're']


In [43]:
from utils.preprocessing import SentenceSplitPreprocessor

prep = SentenceSplitPreprocessor(window=50)
result = prep.process_file("data/raw/UD_English-EWT/en_ewt-ud-train.sent_split")

df = result["df"]
clean_text = result["clean_text"]

print(df.shape)
print(df["label"].value_counts())
print(clean_text[:1000])
print("newline-only boundaries:", len(result["newline_boundary_indices"]))

(13649, 9)
label
1    10164
0     3485
Name: count, dtype: int64
Al-Zaman : American forces killed Shaikh Abdullah al-Ani, the preacher at the
mosque in the town of Qaim, near the Syrian border.
 [This killing of a respected
cleric will be causing us trouble for years to come.]
 DPA: Iraqi authorities
announced that they had busted up 3 terrorist cells operating in Baghdad.
 Two of
them were being run by 2 officials of the Ministry of the Interior!
 The MoI in
Iraq is equivalent to the US FBI, so this would be like having J. Edgar Hoover
unwittingly employ at a high level members of the Weathermen bombers back in the
1960s.
 The third was being run by the head of an investment firm.
 You wonder if
he was manipulating the market with his bombing targets.
 The cells were
operating in the Ghazaliyah and al-Jihad districts of the capital.
 Although the
announcement was probably made to show progress in identifying and breaking up
terror cells, I don't find the news that the Baathists conti

In [44]:
preview = df.sample(12).copy()
preview["left_context"] = preview["left_context"].str.replace("\n", "\\n", regex=False)
preview["right_context"] = preview["right_context"].str.replace("\n", "\\n", regex=False)

preview[["punct", "left_context", "right_context", "label"]]

,punct,left_context,right_context,label
10635,.,er is starting ballet this year for the first time,\n I'ma soccer mom so\nI wasn't sure what I was look,1
3324,.,IC\n3067. Ph. (03) 9221 4101 Fax.(03) 9221 4193 Mob,0417 575 920\n <<Enron - UC\nMaster Agreement - 030,0
13362,.,structor and\nSimply the Best Instructor there is .,"..very calm, pleasant and very detailed in\ngiving",0
9460,.,u something a little different.\n \n\nyoga and horses,....?\n \n\nI got a riding lesson today on one of my,0
9225,.,will get in three business days or your money back,"\n This is in\nOntario of course, the rates may be s",1
11217,.,nd it passed the\nemissions test with flying colors,"\n If you're a fan of herpes, being ripped off,\nand",1
11928,!,nd Holly to anyone.\n You will not be disappointed!,!!!!\n \n\nOutdated but not bad\n \n\nSo I really felt l,0
11819,.,s talking to his\nfriends rather than his customers,\n The angry lobster was completely over-priced!\n\n$,1
7332,.,"lipped pixels, some only show up on long\nexposures",\n Chris\n \n\nNew Zealand skilled migrant visa?\n \n\nIf,1
12051,!,"e street and asked ""Who does your\nhair...I LOVE it","!!""\n Quaint, lovely, small salon with BIG personal",0


In [54]:
print(df.shape)
print(df["label"].value_counts())
print(df["punct"].value_counts())

(13649, 9)
label
1    10164
0     3485
Name: count, dtype: int64
punct
.    11491
!     1221
?      937
Name: count, dtype: int64


In [50]:
from utils.preprocessing import SentenceSplitPreprocessor
from utils.featureExtractor import FeatureExtractor

prep = SentenceSplitPreprocessor(window=50)
result = prep.process_file("data/raw/UD_English-EWT/en_ewt-ud-train.sent_split")

df = result["df"]

feat_extractor = FeatureExtractor()
X_feat = feat_extractor.fit_transform(df)

X_feat.head()

,is_period,is_question,is_exclamation,prev_is_digit,next_is_digit,prev_is_alpha,next_is_alpha,prev_is_space,next_is_space,prev_is_upper,next_is_upper,prev_is_lower,next_is_lower,next_is_newline,token_before_len,token_after_len,token_before_is_short,token_before_is_upper,token_before_is_title,token_after_is_upper,token_after_is_title,token_before_has_digit,token_after_has_digit,is_decimal_pattern,is_possible_abbrev,next_token_starts_upper,leading_quotes_after,leading_closing_brackets_after,has_quote_after,has_closing_bracket_after,prev_is_punct,next_is_punct,trailing_punct_before,leading_punct_after,newline_in_left_10,newline_in_right_10,double_newline_in_right_20
0,1,0,0,0,0,1,0,0,1,0,0,1,0,1,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,1,0,0,0,0,1,0,0,0,0,0,1,0,0,4,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,1,0
2,1,0,0,0,0,1,0,0,1,0,0,1,0,1,7,3,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0
3,0,0,1,0,0,1,0,0,1,0,0,1,0,1,8,3,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0
4,1,0,0,0,0,1,0,0,1,1,0,0,0,0,1,5,1,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0


In [51]:
from utils.preprocessing import SentenceSplitPreprocessor
from utils.text_features import TfidfFeatureExtractor

prep = SentenceSplitPreprocessor(window=50)

train_result = prep.process_file("data/raw/UD_English-EWT/en_ewt-ud-train.sent_split")
dev_result = prep.process_file("data/raw/UD_English-EWT/en_ewt-ud-dev.sent_split")

train_df = train_result["df"]
dev_df = dev_result["df"]

y_train = train_df["label"]
y_dev = dev_df["label"]

tfidf_extractor = TfidfFeatureExtractor(
    text_col="centered_context",
    analyzer="char",
    ngram_range=(2, 6),
    min_df=2,
    lowercase=False,
    use_chi2=False
)

X_train_tfidf = tfidf_extractor.fit_transform(train_df, y_train)
X_dev_tfidf = tfidf_extractor.transform(dev_df)

print(X_train_tfidf.shape)
print(X_dev_tfidf.shape)

(13649, 351653)
(2294, 351653)


In [52]:
tfidf_chi2_extractor = TfidfFeatureExtractor(
    text_col="centered_context",
    analyzer="char",
    ngram_range=(2, 6),
    min_df=2,
    lowercase=False,
    use_chi2=True,
    k_best=8000
)

X_train_tfidf_chi2 = tfidf_chi2_extractor.fit_transform(train_df, y_train)
X_dev_tfidf_chi2 = tfidf_chi2_extractor.transform(dev_df)

print(X_train_tfidf_chi2.shape)
print(X_dev_tfidf_chi2.shape)

(13649, 8000)
(2294, 8000)


In [53]:
feature_names = tfidf_chi2_extractor.get_feature_names_out()
print(feature_names[:50])
print(len(feature_names))

['\n!' '\n!!' '\n!!!' '\n!!!!' '\n!!!!!' '\n(' '\n(4' '\n(41' '\n(415'
 '\n(415.' '\n(7' '\n(71' '\n(713' '\n(713)' '\n202' '\n202.' '\n202.4'
 '\n7' '\n71' '\n713' '\n713-' '\n713-6' '\n<' '\n<<' '\n<<l' '\n<<ld'
 '\n<<ld2' '\n<y' '\n<ym' '\n<yms' '\nAnd' '\nAnd.' '\nAnd..' '\nCu'
 '\nCus' '\nCust' '\nCusto' '\nIt.' '\nIt. ' '\nIt. A' '\nSh' '\nShr'
 '\nShri' '\nShrim' '\nThi ' '\nThi T' '\nalt' '\nalt.' '\nalt.a'
 '\nappre']
8000


### chi2 features

the selected features are char n-grams.

so each one is basically a small substring pattern around the punctuation

### key observation

most selected features start with:

`\n`

this means newline is a very strong signal for sentence boundaries

### examples

**`\nAnd`**  
→ new sentence starting with "And"  
→ strong eos pattern

**`\nIt. A`**  
→ something like:  
`... \nIt. A new sentence`  
→ newline + uppercase after punctuation  
→ strong boundary cue

**`\n202.` , `\n713-`**  
→ numeric / structured patterns at sentence start  
→ often from lists, headings, or formatted text  
→ shows structure matters, not just words

**`\n!!`, `\n!!!`**  
→ expressive punctuation after newline  
→ very likely sentence endings

**`\n(`, `\n(415.`**  
→ sentences starting with parentheses  
→ common in references, citations, structured text

**`\n<<ld2`, `\n<yms`**  
→ dataset-specific patterns / artifacts  
→ model picks them because they correlate with labels

model is heavily relying on newline + local structure, not just the punctuation itself

### Simple LogReg with/out chi

In [62]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

clf.fit(X_train_tfidf, y_train)
dev_pred = clf.predict(X_dev_tfidf)

print(classification_report(y_dev, dev_pred, digits=4))
print("f1:", f1_score(y_dev, dev_pred))

              precision    recall  f1-score   support

           0     0.8549    0.7362    0.7911       800
           1     0.8685    0.9331    0.8996      1494

    accuracy                         0.8644      2294
   macro avg     0.8617    0.8347    0.8454      2294
weighted avg     0.8638    0.8644    0.8618      2294

f1: 0.8996450467892869


In [61]:
clf_chi2 = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

clf_chi2.fit(X_train_tfidf_chi2, y_train)
dev_pred_chi2 = clf_chi2.predict(X_dev_tfidf_chi2)

print(classification_report(y_dev, dev_pred_chi2, digits=4))
print("f1:", f1_score(y_dev, dev_pred_chi2))

              precision    recall  f1-score   support

           0     0.8079    0.8150    0.8114       800
           1     0.9005    0.8963    0.8984      1494

    accuracy                         0.8679      2294
   macro avg     0.8542    0.8556    0.8549      2294
weighted avg     0.8682    0.8679    0.8680      2294

f1: 0.8983562562898356


In [60]:
probs_log = clf.predict_proba(X_dev_tfidf)
probs_log_chi = clf_chi2.predict_proba(X_dev_tfidf_chi2)

blended_probs = (probs_log + probs_log_chi) / 2.0

final_predictions = np.argmax(blended_probs, axis=1)

# I evaluate the soft voting ensemble
print("\n" + "="*60)
print("SOFT VOTING ENSEMBLE PERFORMANCE")
print("="*60)
f1_soft = f1_score(y_dev, final_predictions, average='macro')
print(f"F1 Macro Score (Soft Voting): {f1_soft:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(y_dev, final_predictions, digits=4))


SOFT VOTING ENSEMBLE PERFORMANCE
F1 Macro Score (Soft Voting): 0.8525

Detailed Classification Report:
              precision    recall  f1-score   support

           0     0.8356    0.7750    0.8042       800
           1     0.8840    0.9183    0.9009      1494

    accuracy                         0.8684      2294
   macro avg     0.8598    0.8467    0.8525      2294
weighted avg     0.8671    0.8684    0.8671      2294



In [63]:
lgbm_chi = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary",
    random_state=42,
    n_jobs=-1
)

lgbm_chi.fit(X_train_tfidf_chi2, y_train)

dev_pred_chi = lgbm_chi.predict(X_dev_tfidf_chi2)
dev_prob_chi = lgbm_chi.predict_proba(X_dev_tfidf_chi2)[:, 1]

print("LIGHTGBM + TFIDF (with chi)")
print(classification_report(y_dev, dev_pred_chi, digits=4))
print("f1:", f1_score(y_dev, dev_pred_chi))

[LightGBM] [Info] Number of positive: 10164, number of negative: 3485
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.149097 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 152881
[LightGBM] [Info] Number of data points in the train set: 13649, number of used features: 3489
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.744670 -> initscore=1.070384
[LightGBM] [Info] Start training from score 1.070384
LIGHTGBM + TFIDF (with chi)
              precision    recall  f1-score   support

           0     0.8879    0.6238    0.7327       800
           1     0.8262    0.9578    0.8872      1494

    accuracy                         0.8413      2294
   macro avg     0.8571    0.7908    0.8100      2294
weighted avg     0.8477    0.8413    0.8333      2294

f1: 0.8871667699938004


c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [66]:
svm_no_chi = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

svm_no_chi.fit(X_train_tfidf, y_train)
dev_pred_svm_no_chi = svm_no_chi.predict(X_dev_tfidf)

print("LINEAR SVM + TFIDF (no chi)")
print(classification_report(y_dev, dev_pred_svm_no_chi, digits=4))
print("f1:", f1_score(y_dev, dev_pred_svm_no_chi))

LINEAR SVM + TFIDF (no chi)
              precision    recall  f1-score   support

           0     0.8663    0.6725    0.7572       800
           1     0.8434    0.9444    0.8911      1494

    accuracy                         0.8496      2294
   macro avg     0.8549    0.8085    0.8241      2294
weighted avg     0.8514    0.8496    0.8444      2294

f1: 0.8910640985159457


In [67]:
svm_chi = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

svm_chi.fit(X_train_tfidf_chi2, y_train)
dev_pred_svm_chi = svm_chi.predict(X_dev_tfidf_chi2)

print("LINEAR SVM + TFIDF (with chi)")
print(classification_report(y_dev, dev_pred_svm_chi, digits=4))
print("f1:", f1_score(y_dev, dev_pred_svm_chi))

LINEAR SVM + TFIDF (with chi)
              precision    recall  f1-score   support

           0     0.8025    0.8688    0.8343       800
           1     0.9265    0.8855    0.9055      1494

    accuracy                         0.8797      2294
   macro avg     0.8645    0.8771    0.8699      2294
weighted avg     0.8833    0.8797    0.8807      2294

f1: 0.9055441478439425


### So to ensemble, I will go with LinearSVM+ chi and LogReg without chi since I think they represent different error types and characteristic of the data.

## Optuna

I will use bayesian optimization to find better parameters for the both models.

In [70]:
# Importing optuna

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def optuna_status_callback(study, trial):
    if trial.number % 5 == 0 or trial.number == study.best_trial.number:
        print(
            f"[trial {trial.number:03d}] "
            f"current={trial.value:.5f} | "
            f"best={study.best_value:.5f} | "
            f"best_params={study.best_params}"
        )

In [71]:
def objective_svc_chi(trial):
    ngram_range = trial.suggest_categorical(
        "ngram_range",
        [(2, 5), (2, 6), (3, 6), (3, 7)]
    )
    min_df = trial.suggest_categorical("min_df", [1, 2, 3, 5])
    lowercase = trial.suggest_categorical("lowercase", [False, True])
    k_best = trial.suggest_categorical("k_best", [2000, 4000, 8000, 12000, 20000])

    C = trial.suggest_float("C", 1e-2, 20.0, log=True)
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
    loss = trial.suggest_categorical("loss", ["hinge", "squared_hinge"])

    extractor = TfidfFeatureExtractor(
        text_col="centered_context",
        analyzer="char",
        ngram_range=ngram_range,
        min_df=min_df,
        lowercase=lowercase,
        use_chi2=True,
        k_best=k_best
    )

    X_train = extractor.fit_transform(train_df, y_train)
    X_dev = extractor.transform(dev_df)

    clf = LinearSVC(
        C=C,
        class_weight=class_weight,
        loss=loss,
        max_iter=10000,
        random_state=42
    )

    clf.fit(X_train, y_train)
    pred = clf.predict(X_dev)

    score = f1_score(y_dev, pred, average="macro")
    return score

In [72]:
study_svc = optuna.create_study(
    study_name="svc_chi_macro_f1",
    direction="maximize"
)

study_svc.optimize(
    objective_svc_chi,
    n_trials=40,
    show_progress_bar=True,
    callbacks=[optuna_status_callback]
)

print("best value:", study_svc.best_value)
print("best params:", study_svc.best_params)

  0%|          | 0/40 [00:00<?, ?it/s]

c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 5) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (3, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be 

[trial 000] current=0.80254 | best=0.80254 | best_params={'ngram_range': (3, 6), 'min_df': 1, 'lowercase': True, 'k_best': 2000, 'C': 0.30081990964156746, 'class_weight': None, 'loss': 'squared_hinge'}


c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 5) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (3, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be 

[trial 001] current=0.87116 | best=0.87116 | best_params={'ngram_range': (2, 5), 'min_df': 5, 'lowercase': True, 'k_best': 2000, 'C': 0.795769261867023, 'class_weight': 'balanced', 'loss': 'squared_hinge'}


...

[trial 030] current=0.84735 | best=0.87598 | best_params={'ngram_range': (2, 6), 'min_df': 1, 'lowercase': False, 'k_best': 8000, 'C': 7.113084250627605, 'class_weight': 'balanced', 'loss': 'squared_hinge'}


c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 5) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (3, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be 

[trial 035] current=0.87034 | best=0.87598 | best_params={'ngram_range': (2, 6), 'min_df': 1, 'lowercase': False, 'k_best': 8000, 'C': 7.113084250627605, 'class_weight': 'balanced', 'loss': 'squared_hinge'}


c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 5) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (3, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be 

best value: 0.8759792859420796
best params: {'ngram_range': (2, 6), 'min_df': 1, 'lowercase': False, 'k_best': 8000, 'C': 7.113084250627605, 'class_weight': 'balanced', 'loss': 'squared_hinge'}


In [73]:
def objective_logreg(trial):
    ngram_range = trial.suggest_categorical(
        "ngram_range",
        [(2, 5), (2, 6), (3, 6), (3, 7)]
    )
    min_df = trial.suggest_categorical("min_df", [1, 2, 3, 5])
    lowercase = trial.suggest_categorical("lowercase", [False, True])

    solver = trial.suggest_categorical("solver", ["liblinear"])
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    C = trial.suggest_float("C", 1e-3, 30.0, log=True)
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])

    extractor = TfidfFeatureExtractor(
        text_col="centered_context",
        analyzer="char",
        ngram_range=ngram_range,
        min_df=min_df,
        lowercase=lowercase,
        use_chi2=False
    )

    X_train = extractor.fit_transform(train_df, y_train)
    X_dev = extractor.transform(dev_df)

    clf = LogisticRegression(
        solver=solver,
        penalty=penalty,
        C=C,
        class_weight=class_weight,
        max_iter=5000,
        random_state=42
    )

    clf.fit(X_train, y_train)
    pred = clf.predict(X_dev)

    score = f1_score(y_dev, pred, average="macro")
    return score

In [74]:
study_logreg = optuna.create_study(
    study_name="logreg_macro_f1",
    direction="maximize"
)

study_logreg.optimize(
    objective_logreg,
    n_trials=40,
    show_progress_bar=True,
    callbacks=[optuna_status_callback]
)

print("best value:", study_logreg.best_value)
print("best params:", study_logreg.best_params)

  0%|          | 0/40 [00:00<?, ?it/s]

c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 5) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (3, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be 

[trial 000] current=0.81097 | best=0.81097 | best_params={'ngram_range': (2, 6), 'min_df': 1, 'lowercase': False, 'solver': 'liblinear', 'penalty': 'l2', 'C': 0.0117986684846629, 'class_weight': 'balanced'}


c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 5) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (3, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be 

[trial 001] current=0.82271 | best=0.82271 | best_params={'ngram_range': (2, 5), 'min_df': 3, 'lowercase': False, 'solver': 'liblinear', 'penalty': 'l2', 'C': 0.015768107276771765, 'class_weight': 'balanced'}


...

[trial 033] current=0.87528 | best=0.87528 | best_params={'ngram_range': (2, 6), 'min_df': 1, 'lowercase': False, 'solver': 'liblinear', 'penalty': 'l1', 'C': 4.056988069254317, 'class_weight': 'balanced'}


c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 5) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (3, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be 

[trial 035] current=0.39440 | best=0.87528 | best_params={'ngram_range': (2, 6), 'min_df': 1, 'lowercase': False, 'solver': 'liblinear', 'penalty': 'l1', 'C': 4.056988069254317, 'class_weight': 'balanced'}


c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 5) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (2, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (3, 6) which is of type tuple.
  optuna_warn(message)
c:\Users\anili\miniconda3\envs\sent-split\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be 

best value: 0.8752846034926622
best params: {'ngram_range': (2, 6), 'min_df': 1, 'lowercase': False, 'solver': 'liblinear', 'penalty': 'l1', 'C': 4.056988069254317, 'class_weight': 'balanced'}


In [ ]:
optuna.visualization.plot_optimization_history(study_svc)

In [77]:
optuna.visualization.plot_param_importances(study_svc)

In [78]:
optuna.visualization.plot_optimization_history(study_logreg)

In [79]:
optuna.visualization.plot_param_importances(study_logreg)

## Ensemble with Calibration + Stacking 

### why calibration is needed here

StackingClassifier with stack_method="predict_proba" expects probability outputs.

logistic regression already has predict_proba
LinearSVC does not

So SVC must be wrapped with CalibratedClassifierCV.

For consistency, and to compare fair soft-voting vs stacking later, I would calibrate both base models.

Also, I would start with method="sigmoid" rather than isotonic. It is usually the safer first choice here.

Best parameters are found by optuna


In [80]:
svc_chi_params = {
    "tfidf__text_col": "centered_context",
    "tfidf__analyzer": "char",
    "tfidf__ngram_range": (2, 6),
    "tfidf__min_df": 1,
    "tfidf__lowercase": False,
    "tfidf__use_chi2": True,
    "tfidf__k_best": 8000,

    "clf__C": 7.113084250627605,
    "clf__class_weight": "balanced",
    "clf__loss": "squared_hinge",
    "clf__max_iter": 10000,
    "clf__random_state": 42
}

logreg_params = {
    "tfidf__text_col": "centered_context",
    "tfidf__analyzer": "char",
    "tfidf__ngram_range": (2, 6),
    "tfidf__min_df": 1,
    "tfidf__lowercase": False,
    "tfidf__use_chi2": False,

    "clf__solver": "liblinear",
    "clf__penalty": "l1",
    "clf__C": 4.056988069254317,
    "clf__class_weight": "balanced",
    "clf__max_iter": 5000,
    "clf__random_state": 42
}

In [ ]:
svc_chi_pipe = Pipeline([
    ("tfidf", TfidfFeatureExtractor()),
    ("clf", LinearSVC())
])
svc_chi_pipe.set_params(**svc_chi_params)


logreg_pipe = Pipeline([
    ("tfidf", TfidfFeatureExtractor()),
    ("clf", LogisticRegression())
])
logreg_pipe.set_params(**logreg_params)

# base estimators with calibration
estimators = [
    (
        "svc_chi",
        CalibratedClassifierCV(
            estimator=svc_chi_pipe,
            method="sigmoid",
            cv=5
        )
    ),
    (
        "logreg",
        CalibratedClassifierCV(
            estimator=logreg_pipe,
            method="sigmoid",
            cv=5
        )
    ),
]

# I define the meta-learner: Simple Logistic Regression
# This learns optimal weights for combining the two base models
meta = LogisticRegression(
    class_weight="balanced", # Handle potential class imbalance in meta-features
    C=1.0, 
    max_iter=2000,
    random_state=42
)

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=meta,
    cv=5,
    stack_method="predict_proba",
    passthrough=False,  # Do NOT include original features in meta-features
    n_jobs=-1
)

print("training stacking classifier with calibration...")
stacking_clf.fit(train_df, y_train)
print("✓ stacking classifier training completed")

y_pred_stack = stacking_clf.predict(dev_df)

print("\n" + "=" * 60)
print("STACKING ENSEMBLE PERFORMANCE")
print("=" * 60)
f1_stack = f1_score(y_dev, y_pred_stack, average="macro")
print(f"F1 Macro Score (Stacking): {f1_stack:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(y_dev, y_pred_stack, digits=4))

training stacking classifier with calibration...
✓ stacking classifier training completed

STACKING ENSEMBLE PERFORMANCE
F1 Macro Score (Stacking): 0.8782

Detailed Classification Report:
              precision    recall  f1-score   support

           0     0.7978    0.9025    0.8469       800
           1     0.9438    0.8775    0.9095      1494

    accuracy                         0.8862      2294
   macro avg     0.8708    0.8900    0.8782      2294
weighted avg     0.8929    0.8862    0.8877      2294



In [83]:
estimators2 = [
    (
        "svc_chi",
        CalibratedClassifierCV(
            estimator=svc_chi_pipe,
            method="isotonic",
            cv=5
        )
    ),
    (
        "logreg",
        CalibratedClassifierCV(
            estimator=logreg_pipe,
            method="isotonic",
            cv=5
        )
    ),
]

meta2 = LogisticRegression(
    class_weight="balanced", 
    C=0.1, # Regularization strength (higher = less regularization)
    max_iter=1000,
    random_state=42
)

stacking_clf2 = StackingClassifier(
    estimators=estimators2,
    final_estimator=meta2,
    cv=5,
    stack_method="predict_proba",
    passthrough=False,
    n_jobs=-1
)

print("training stacking classifier with calibration...")
stacking_clf2.fit(train_df, y_train)
print("✓ stacking classifier training completed")

y_pred_stack2 = stacking_clf2.predict(dev_df)

print("\n" + "=" * 60)
print("STACKING ENSEMBLE PERFORMANCE")
print("=" * 60)
f1_stack = f1_score(y_dev, y_pred_stack2, average="macro")
print(f"F1 Macro Score (Stacking): {f1_stack:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(y_dev, y_pred_stack2, digits=4))

training stacking classifier with calibration...
✓ stacking classifier training completed

STACKING ENSEMBLE PERFORMANCE
F1 Macro Score (Stacking): 0.8715

Detailed Classification Report:
              precision    recall  f1-score   support

           0     0.7648    0.9387    0.8429       800
           1     0.9627    0.8454    0.9002      1494

    accuracy                         0.8779      2294
   macro avg     0.8637    0.8921    0.8715      2294
weighted avg     0.8936    0.8779    0.8802      2294

